In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

custom_llm = ChatOpenAI(
    model="minimax-local",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("CHAT_API_BASE_URL"),
    temperature=0,
    verbose=True
)

In [10]:
from langchain_core.tools import tool

@tool
def get_weather(city:str) -> str:
    
    """Get the current weather for a given city."""
    return f"The current weather in {city} is sunny."

model_with_tools=custom_llm.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 197, 'total_tokens': 253, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/minimax-local', 'system_fingerprint': None, 'id': 'chatcmpl-8c15ceca842a8c58', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019eb81b-5077-7990-a01a-d22a0b23f2fc-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'chatcmpl-tool-a360a960f6ca8dbc', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 197, 'output_tokens': 56, 'total_tokens': 253, 'input_token_details': {}, 'output_token_details': {}}


In [7]:
from langchain_core.messages import HumanMessage, SystemMessage

response = model_with_tools.invoke([
    SystemMessage(content="Use get_weather tool to find the weather"),
    HumanMessage(content="What is the weather in New York?")
])
print(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 185, 'total_tokens': 237, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/minimax-local', 'system_fingerprint': None, 'id': 'chatcmpl-b90930e90d9eb395', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019eb81b-54ad-7891-8c67-231c5ab1d858-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'chatcmpl-tool-ba8d2efd9a44ead7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 185, 'output_tokens': 52, 'total_tokens': 237, 'input_token_details': {}, 'output_token_details': {}}


In [10]:
response = model_with_tools.invoke("What is the weather in New York? Use get_weather tool to find the weather")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with arguments {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 206, 'total_tokens': 277, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/minimax-local', 'system_fingerprint': None, 'id': 'chatcmpl-a8ed03547e22c344', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019eb81e-41e2-77c0-90ae-f84f4a5a52c4-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'New York'}, 'id': 'chatcmpl-tool-a63ff56d9c3109fb', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 206, 'output_tokens': 71, 'total_tokens': 277, 'input_token_details': {}, 'output_token_details': {}}
Tool called: get_weather with arguments {'city': 'New York'}


### Tool Execution Loop

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72°F and sunny."




The current weather in Boston is sunny.


In [13]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 195, 'total_tokens': 248, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hosted_vllm/minimax-local', 'system_fingerprint': None, 'id': 'chatcmpl-b65ec0b0889719d5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb823-3c1e-72d3-ab6a-d23f9c93356d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'chatcmpl-tool-a8906c0a4297973a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 53, 'total_tokens': 248, 'input_token_details': {}, 'output_token_details': {}}),
 ToolMessage(content='The current weather in Boston is sunny.', name='get_weather', tool_call_id='chatcmpl-tool-a8906c0a4297973a')]

### Different message types